# Latin American Sovereign External Debt Risk

**Goal:** Identify which Latin American & Caribbean (LAC) countries carry the most external debt risk, using World Bank International Debt Statistics indicators, real SQL queries, and a Power BI dashboard.

**Why this project:** External debt sustainability is a core topic in development finance and capital markets -- countries with high short-term debt exposure or fast-rising debt service burdens face real refinancing risk, which is exactly the kind of thing an investment officer, capital markets analyst, or macro/development economist tracks.

**Skills demonstrated:** Python (data pull), SQL (real queries, not just pandas), Power BI (dashboard).

**Pipeline:**
1. Setup
2. Pull debt & macro data from the World Bank
3. Load into a SQL database (SQLite)
4. Write SQL queries to answer analyst questions
5. Export a clean table for Power BI
6. Build the Power BI dashboard (instructions below -- done outside this notebook, in Power BI Service)

> Run this in Google Colab or local Jupyter with internet access.

In [ ]:
!pip install wbgapi --quiet

import wbgapi as wb
import pandas as pd
import numpy as np
import sqlite3

pd.set_option('display.max_columns', None)

## 2. Pull data

**Indicators (World Bank International Debt Statistics + WDI):**
- `DT.DOD.DECT.GN.ZS` -- external debt stocks (% of GNI)
- `DT.TDS.DECT.EX.ZS` -- total debt service (% of exports of goods, services and primary income)
- `DT.DOD.DSTC.ZS` -- short-term external debt (% of total external debt) -- a refinancing-risk signal
- `NY.GDP.MKTP.KD.ZG` -- GDP growth (annual %) -- macro context
- `FP.CPI.TOTL.ZG` -- inflation, consumer prices (annual %) -- macro context

**Note:** IDS data usually lags 1-2 years behind the current date, and is only reported for developing economies (so a few smaller LAC countries may be missing -- that's normal, not a bug).

In [ ]:
lac_countries = [
    'ARG', 'BOL', 'BRA', 'CHL', 'COL', 'CRI', 'DOM', 'ECU', 'SLV',
    'GTM', 'GUY', 'HTI', 'HND', 'JAM', 'MEX', 'NIC', 'PAN', 'PRY',
    'PER', 'SUR', 'URY'
]

indicators = {
    'DT.DOD.DECT.GN.ZS': 'debt_pct_gni',
    'DT.TDS.DECT.EX.ZS': 'debt_service_pct_exports',
    'DT.DOD.DSTC.ZS': 'short_term_debt_pct',
    'NY.GDP.MKTP.KD.ZG': 'gdp_growth_pct',
    'FP.CPI.TOTL.ZG': 'inflation_pct',
}

years = range(2010, 2023)

# sanity check if a code has been renamed/retired:
# wb.series.info(q='external debt')

raw = wb.data.DataFrame(
    list(indicators.keys()),
    economy=lac_countries,
    time=years,
    skipBlanks=True,
).reset_index()

raw.head()

,economy,series,YR2010,YR2011,YR2012,YR2013,YR2014,YR2015,YR2016,YR2017,YR2018,YR2019,YR2020,YR2021,YR2022
0,ARG,DT.DOD.DECT.GN.ZS,30.949130,27.744178,26.239688,27.809745,29.831902,30.359713,33.312561,36.021014,54.896071,65.249945,68.098340,51.617688,40.082194
1,ARG,DT.DOD.DSTC.ZS,12.960200,18.018500,18.052900,24.424500,22.036800,33.543800,22.581600,24.568700,24.393000,23.876000,16.669100,17.770200,19.821000
2,ARG,DT.TDS.DECT.EX.ZS,18.642455,15.654928,14.066678,17.137319,20.169582,24.733476,34.928014,51.729912,51.548258,50.832792,43.097264,30.602265,31.682590
3,ARG,FP.CPI.TOTL.ZG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,34.277224,53.548304,42.015095,48.409379,72.430758
4,ARG,NY.GDP.MKTP.KD.ZG,10.125398,6.003952,-1.026420,2.405324,-2.512615,2.731160,-2.080328,2.818503,-2.617396,-2.000861,-9.900485,10.441812,6.020745


## 3. Load into a SQL database

Reshape into a tidy long panel, then load it into an actual SQLite database -- this is what lets us write real SQL queries instead of just pandas.

In [ ]:
year_cols = [c for c in raw.columns if c.startswith('YR')]
long = raw.melt(id_vars=['economy', 'series'], value_vars=year_cols,
                var_name='year', value_name='value')
long['year'] = long['year'].str.replace('YR', '').astype(int)

wide = long.pivot_table(index=['economy', 'year'], columns='series', values='value').reset_index()
wide = wide.rename(columns=indicators)

print('Missing values per column:')
print(wide.isna().sum())

# Create SQLite database and load the table
conn = sqlite3.connect('debt_data.db')
wide.to_sql('debt_panel', conn, if_exists='replace', index=False)

print('\nLoaded', len(wide), 'rows into debt_panel table.')

Missing values per column:
series
economy                      0
year                         0
debt_pct_gni                57
short_term_debt_pct         57
debt_service_pct_exports    57
inflation_pct               10
gdp_growth_pct               0
dtype: int64

Loaded 273 rows into debt_panel table.


## 4. SQL queries -- answering real analyst questions

**Query 1: Most recent snapshot -- which countries currently have the highest external debt burden?**

In [ ]:
query1 = """
SELECT economy, year, debt_pct_gni, debt_service_pct_exports, short_term_debt_pct
FROM debt_panel
WHERE year = (SELECT MAX(year) FROM debt_panel WHERE debt_pct_gni IS NOT NULL)
  AND debt_pct_gni IS NOT NULL
ORDER BY debt_pct_gni DESC
LIMIT 10;
"""
pd.read_sql(query1, conn)

,economy,year,debt_pct_gni,debt_service_pct_exports,short_term_debt_pct
0,SUR,2022,120.403536,14.714590,9.8281
1,NIC,2022,104.763776,23.716629,7.5560
2,JAM,2022,94.432157,22.610524,13.9354
3,SLV,2022,70.809777,32.150134,11.1788
4,PRY,2022,62.194605,14.082482,9.1760
5,COL,2022,55.290160,33.803973,10.4956
6,ECU,2022,53.129747,14.439475,3.1693
7,DOM,2022,44.234494,14.186344,6.6563
8,HND,2022,43.702488,17.108382,7.8278
9,MEX,2022,40.897191,8.612037,9.3480


**Query 2: Which countries' debt service burden has grown fastest over the last 5 years of available data?**

This uses a SQL window function (`LAG`) to compute year-over-year change -- a genuinely useful SQL technique for time-series data, not just a basic SELECT.

In [ ]:
query2 = """
WITH yoy AS (
    SELECT
        economy,
        year,
        debt_service_pct_exports,
        debt_service_pct_exports - LAG(debt_service_pct_exports) OVER (
            PARTITION BY economy ORDER BY year
        ) AS yoy_change
    FROM debt_panel
    WHERE debt_service_pct_exports IS NOT NULL
)
SELECT economy, year, debt_service_pct_exports, yoy_change
FROM yoy
WHERE year >= (SELECT MAX(year) FROM debt_panel) - 5
  AND yoy_change IS NOT NULL
ORDER BY yoy_change DESC
LIMIT 10;
"""
pd.read_sql(query2, conn)

,economy,year,debt_service_pct_exports,yoy_change
0,JAM,2019,86.041429,62.209952
1,SLV,2017,51.621341,31.556487
2,DOM,2020,40.418459,23.006643
3,SLV,2019,67.753909,22.221259
4,BRA,2019,53.470277,20.160834
5,COL,2020,50.768336,17.803401
6,ARG,2017,51.729912,16.801898
7,SLV,2020,82.029397,14.275488
8,HND,2020,23.826912,11.831291
9,GTM,2020,25.376577,11.408770


**Query 3: Refinancing risk flag -- countries where short-term debt makes up a large share of total external debt (a classic early-warning signal)**

In [ ]:
query3 = """
SELECT economy, year, short_term_debt_pct,
    CASE
        WHEN short_term_debt_pct >= 15 THEN 'Elevated risk'
        WHEN short_term_debt_pct >= 8 THEN 'Moderate risk'
        ELSE 'Lower risk'
    END AS risk_flag
FROM debt_panel
WHERE year = (SELECT MAX(year) FROM debt_panel WHERE short_term_debt_pct IS NOT NULL)
  AND short_term_debt_pct IS NOT NULL
ORDER BY short_term_debt_pct DESC;
"""
pd.read_sql(query3, conn)

,economy,year,short_term_debt_pct,risk_flag
0,ARG,2022,19.8210,Elevated risk
1,JAM,2022,13.9354,Moderate risk
2,HTI,2022,12.6630,Moderate risk
3,BRA,2022,11.6997,Moderate risk
4,PER,2022,11.4810,Moderate risk
5,SLV,2022,11.1788,Moderate risk
6,COL,2022,10.4956,Moderate risk
7,SUR,2022,9.8281,Moderate risk
8,MEX,2022,9.3480,Moderate risk
9,PRY,2022,9.1760,Moderate risk


**Query 4: Does higher debt correlate with lower growth? A simple join-based check.**

In [ ]:
query4 = """
SELECT economy,
       ROUND(AVG(debt_pct_gni), 1) AS avg_debt_pct_gni,
       ROUND(AVG(gdp_growth_pct), 1) AS avg_gdp_growth
FROM debt_panel
GROUP BY economy
HAVING avg_debt_pct_gni IS NOT NULL AND avg_gdp_growth IS NOT NULL
ORDER BY avg_debt_pct_gni DESC;
"""
pd.read_sql(query4, conn)

,economy,avg_debt_pct_gni,avg_gdp_growth
0,SUR,110.1,0.0
1,JAM,103.1,1.2
2,NIC,98.9,3.6
3,SLV,69.8,2.4
4,PRY,51.9,3.6
5,MEX,43.2,1.9
6,ARG,40.2,1.6
7,DOM,39.8,5.2
8,GUY,39.1,12.7
9,HND,39.0,3.4


## 5. Export a clean table for Power BI

Power BI Service (the browser version, no desktop app needed) can upload an Excel file directly and build visuals from it. Export the full panel plus a risk-flag column so the dashboard has everything it needs in one file.

In [ ]:
export_query = """
SELECT *,
    CASE
        WHEN short_term_debt_pct >= 15 THEN 'Elevated risk'
        WHEN short_term_debt_pct >= 8 THEN 'Moderate risk'
        WHEN short_term_debt_pct IS NOT NULL THEN 'Lower risk'
        ELSE NULL
    END AS risk_flag
FROM debt_panel
ORDER BY economy, year;
"""
export_df = pd.read_sql(export_query, conn)
export_df.to_excel('lac_debt_dashboard_data.xlsx', index=False)

print('Exported', len(export_df), 'rows to lac_debt_dashboard_data.xlsx')
print('Download this file from the Colab file browser (folder icon on the left) before continuing.')

Exported 273 rows to lac_debt_dashboard_data.xlsx
Download this file from the Colab file browser (folder icon on the left) before continuing.


## 6. Build the Power BI dashboard

Do this part outside the notebook, in your browser:

1. Download `lac_debt_dashboard_data.xlsx` from Colab (folder icon on the left sidebar -> right-click the file -> Download).
2. Go to **app.powerbi.com** and sign in (a free Microsoft/school account works).
3. In **My Workspace**, click **+ New** -> **Upload a file** -> select your Excel file. This creates a dataset.
4. From that dataset, click **Create report**.
5. Build these visuals (drag fields from the right-hand panel):
   - **Line chart:** `year` on the x-axis, `debt_pct_gni` on the y-axis, `economy` as legend -- shows debt trends over time by country.
   - **Bar chart:** `economy` on the x-axis, `short_term_debt_pct` on the y-axis, filtered to the most recent year -- shows current refinancing risk by country.
   - **Table or matrix:** `economy`, `risk_flag`, `debt_service_pct_exports` -- a scannable risk summary.
   - **Scatter chart:** `avg_debt_pct_gni` (x) vs `gdp_growth_pct` (y) -- visualizes the debt-vs-growth relationship from Query 4.
   - Add a **slicer** on `economy` or `year` so the dashboard is interactive.
6. Save the report with a clear title, e.g. "LAC Sovereign Debt Risk Dashboard."

This is the deliverable you can screenshot, link, or walk through live in an interview.

## 7. Interpretation notes (fill in after building)

- Which countries show up as highest-risk across multiple indicators (high debt-to-GNI *and* high short-term debt share)? Are these the same countries, or does risk look different depending on which indicator you use?
- Is there a visible relationship between average debt burden and average GDP growth in Query 4? Be careful with causation here -- slower-growing economies may borrow more *because* they're struggling, not the other way around.
- **Caveats:** This is a simplified risk screen using public macro indicators, not a full debt sustainability analysis (which would also consider currency composition of debt, maturity profiles, and market access). Frame it as a first-pass screening tool, not a definitive risk rating.